In [ ]:
%load_ext autoreload
%autoreload 2
    
import os
import pandas as pd
import math
import matplotlib.pyplot as plt
import numpy as np
import random

from scipy.special import erfc, gamma, gammaincc
from scipy.integrate import quad
import itertools


In [ ]:
def getFalseHitProb(threshold, ENC):
    p = 0.5*erfc(threshold/(ENC*math.sqrt(2)))
    return p

In [ ]:
print(getFalseHitProb(10,3))

In [ ]:
# Make figure and add plots
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

len = 3
threshold = np.logspace(0, len, 100*len+1)
ENC = [3, 10, 50, 100]

for inENC in ENC:
    p = getFalseHitProb(threshold, inENC)
    ax[0].plot(
        threshold, p,
        label=f'ENC = {inENC}'
    )
    ax[1].plot(
        threshold/inENC, p,
        label=f'ENC = {inENC}'
    )

ax[0].set_xlabel(r'Threshold: $n_\text{th}$')
ax[1].set_xlabel(r'Threshold Factor: $n_\text{th}$ / ENC')

for inAx in ax:
    #inAx.set_xscale('log')
    inAx.set_ylabel('False Hit Probability')
    inAx.grid()
ax[0].legend()
ax[1].set_yscale('log')
ax[1].set_xlim([.9, 11])
ax[1].set_ylim([1e-21, 2])

plt.tight_layout()
plt.show()

In [ ]:
print(getFalseHitProb(24/3))

In [ ]:
def myPolya(n, gain, theta):
    A = 1/gain
    B = np.power(theta+1, theta+1)
    C = 1/gamma(theta+1)
    D = np.power(n/gain, theta)
    E = np.exp(-n/gain*(theta+1))

    result = A*B*C*D*E
    return result

def signalAboveThreshold(x, threshold, gain, theta, ENC):

    probAvalancheSize = myPolya(x, gain, theta)
    num = threshold - x
    denom = np.sqrt(2)*ENC
    probAboveThreshold = 0.5*erfc(num/denom)

    result = probAvalancheSize*probAboveThreshold

    return result

def efficiencyAtThreshold(threshold, gain, theta, ENC):
    efficiency, _ = quad(
        signalAboveThreshold,
        a=0.0,
        b=10*gain,
        args=(threshold, gain, theta, ENC)
    )

    return efficiency

In [ ]:
gain = 50
theta = 1.5
ENC = 3
threshold =10


eff = efficiencyAtThreshold(threshold, gain, theta, ENC)
print(eff)

In [ ]:
thetas = [0, 1, 2]
thresholds = np.linspace(0, gain*2.5, 150)
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

axs[0].axvline(
    gain, 
    c='m', ls='--', lw=2, label=f'Gain = {gain}'
)

for theta in thetas:
    efficiencies = [
        efficiencyAtThreshold(th, gain, theta, ENC) 
        for th in thresholds
    ]
    
    charge = np.linspace(0, gain*2.5, 500)
    polyaProb = myPolya(charge, gain, theta)
    
    
    
    axs[0].plot(
        charge, polyaProb, 
        label=rf'Polya ($\theta$={theta:.1f})'
    )
    '''
    axs[0].fill_between(
        charge, polyaProb, alpha=0.5, color='m'
    )
    
    axs[0].axvline(
        10, 
        c='r', ls='--', lw=2, label=f'Threshold = 10e'
    )

    '''
    
    
    
    
    
    axs[1].plot(
        thresholds, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
'''
axs[1].axvline(
    10, 
    c='r', ls='--', lw=2, label=f'Threshold = 10e'
)'''

axs[0].set_xlabel('Avalanche Size: n')
axs[0].set_ylabel(r'PDF: P(n)')

axs[1].set_xlabel('Threshold (e)')
axs[1].set_ylabel('Detection Efficiency (%)')

for inAx in axs:
    inAx.grid()
    inAx.legend()

plt.tight_layout()
plt.show()

In [ ]:
length = 3
thetas = [0, 1, 2]
thresholds = np.linspace(1, gain*length, 100)
charge = np.linspace(0, gain*length, 101)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))

axs[0].axvline(
    gain/gain, 
    c='m', ls='--', lw=2, label=f'Mean Gain'
)

for theta in thetas:
    efficiencies = [
        efficiencyAtThreshold(th, gain, theta, ENC) 
        for th in thresholds
    ]
    
    
    polyaProb = myPolya(charge, gain, theta)
    
    
    
    axs[0].plot(
        charge/gain, polyaProb*gain, 
        label=rf'Polya ($\theta$={theta:.1f})'
    )
    '''
    axs[0].fill_between(
        charge, polyaProb, alpha=0.5, color='m'
    )
    
    axs[0].axvline(
        10, 
        c='r', ls='--', lw=2, label=f'Threshold = 10e'
    )

    '''
    
    
    
    
    
    axs[1].plot(
        thresholds/gain, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
    axs[2].plot(
        gain/thresholds, efficiencies, 
        label=rf'S-Curve ($\theta$={theta:.1f})'
    )
'''
axs[1].axvline(
    10, 
    c='r', ls='--', lw=2, label=f'Threshold = 10e'
)'''

axs[0].set_xlabel(r'Avalanche Size: n / $\overline{n}$')
axs[0].set_ylabel(r'PDF: $\overline{n}$ * P(n)')

axs[1].set_xlabel('Threshold / Gain : 1/GTR')
axs[1].set_ylabel('Detection Efficiency (%)')

axs[2].set_xlabel('Gain / Threshold: GTR')
axs[2].set_ylabel('Detection Efficiency (%)')

axs[2].set_xscale('log')

for inAx in axs:
    inAx.grid()
    inAx.legend()

plt.tight_layout()
plt.show()

In [ ]:
def myPolyaNormal(nNormal, theta):
    A = np.power(theta+1, theta+1)
    B = 1/gamma(theta+1)
    C = np.power(nNormal, theta)
    D = np.exp(-nNormal*(theta+1))
    polyaPDF = A*B*C*D
    return polyaPDF

def myPolyaEfficiencyNormal(GTR, theta):    
    s = theta+1
    x = s/GTR
    efficiency = gammaincc(s, x)
    return efficiency

def riceNoiseRate(TNR, freq):
    riceRate = freq/math.sqrt(3)*np.exp(-0.5*TNR**2)
    return riceRate

def riceOccupancy(TNR, freq, dt):
    riceRate = riceNoiseRate(TNR, freq)
    probability = -np.expm1(-riceRate*dt)
    return np.clip(probability, 0.0, 1.0)

In [ ]:
thetas = np.array([0, 1, 2])

freq = 100e6 # 100 MHz
dt = 10e-9 #10 ns

n = np.linspace(0, 5, 501)
GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0.5, 7.0, 300)

fig, axs = plt.subplots(2, 2, figsize=(12, 8))
axs = axs.flatten()

# Signal Efficiencies
for th in thetas:

    polyaShape = myPolyaNormal(n, th)
    axs[0].plot(
        n, polyaShape,
        label=rf'$\theta$={th:.1f}'
    )

    polyaEff = myPolyaEfficiencyNormal(GTR, th)
    axs[1].plot(
        GTR, polyaEff,
        label=rf'$\theta$={th:.1f}'
    )    
    axs[2].plot(
        1/GTR, polyaEff,
        label=rf'$\theta$={th:.1f}'
    )

# Noise Occupancies
axs[3].plot(
    TNR, riceOccupancy(TNR, freq, dt),
    c='r', lw=2,
    label=rf'f={freq/1e6:.0f} MHz, dt={dt*1e9:.0f} ns'
)

for ax in axs:
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')
axs[2].set_xscale('log')
axs[3].set_xscale('log')

axs[0].set_xlabel(r'Normalized Gain: $\overline{n} / n$')
axs[1].set_xlabel('Gain to Threshold Ratio: GTR')
axs[2].set_xlabel('Threshold / Gain: 1/GTR')
axs[3].set_xlabel('Threshold / Noise: TNR')

axs[0].set_ylabel(r'Polya PDF: $\overline{n} * P(n)$')
axs[1].set_ylabel('Detection Efficiency')
axs[2].set_ylabel('Detection Efficiency')
axs[3].set_ylabel('Noise Occupancy')


plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 1e8 
dt = 10e-9

GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0, 10, 301)

ployaEff_t1 = myPolyaEfficiencyNormal(GTR, 1)
ployaEff_t2 = myPolyaEfficiencyNormal(GTR, 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax in axs:
    
    ax.fill_between(
        1/GTR, ployaEff_t1, ployaEff_t2,
        color='r', alpha=0.5,
        label=r'Polya Efficiency ($1\leq\theta\leq2$)'
    )
    
    ax.plot(
        TNR, riceOccupancy(TNR, freq, dt),
        c='b', lw=2, label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    ax.set_xlabel('Threshold/Gain (1/GTR) and Threshold/Noise (TNR)')
    
    ax.set_ylabel('Signal Efficiency')
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 1e8 
dt = 10e-9

GTR = np.logspace(-1, 2, 301)
TNR = np.linspace(0.01, 10, 301)

ployaEff_t1 = myPolyaEfficiencyNormal(GTR, 1)
ployaEff_t2 = myPolyaEfficiencyNormal(GTR, 2)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax in axs:
    
    ax.fill_between(
        GTR, ployaEff_t1, ployaEff_t2,
        color='r', alpha=0.5,
        label=r'Polya Efficiency ($1\leq\theta\leq2$)'
    )
    
    ax.plot(
        1/TNR, riceOccupancy(TNR, freq, dt),
        c='b', lw=2, label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    ax.set_xlabel('Gain/Threshold (GTR) and Noise/Threshold (1/TNR)')
    
    ax.set_ylabel('Signal Efficiency')
    ax.grid()
    ax.legend()

axs[1].set_xscale('log')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

ENC = np.array([3, 60])
TNR = 5.0

gain = np.logspace(0, 4, 500)
for ax in axs:
    for noise in ENC:
        threshold = noise*TNR
        
        ax.fill_between(
            gain, 
            myPolyaEfficiencyNormal(gain/threshold, 1), 
            myPolyaEfficiencyNormal(gain/threshold, 2),
            label=rf'Polya Efficiency ($1\leq\theta\leq2$), ENC = {noise}'
        )

    ax.grid()
    ax.legend()
    ax.set_xlabel(r'Gain: $\overline{n}$')
    ax.set_ylabel('Detection Efficiency')

axs[1].set_xscale('log')
plt.tight_layout()

In [ ]:
# Setup
freq = 100e6
dt = 100e-9

ENC = np.array([3, 60])
gains = [1e2, 1e3]

# Sweep Threshold in Noise Sigmas (TNR = Q_th / ENC)
TNR = np.logspace(-1, 3, 501)

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax, gain in zip(axs, gains):
    ax.plot(
        TNR, 
        riceOccupancy(TNR, freq, dt),
        color='r', ls='--', lw=2.5,
        label=f'Rice Noise Occupancy (f={freq/1e6:.0f} MHz, dt={dt/1e-9:.0f} ns)'
    )

    for noise in ENC:
        GTR = gain/(TNR*noise)
        ax.fill_between(
            TNR, 
            myPolyaEfficiencyNormal(GTR, 1), 
            myPolyaEfficiencyNormal(GTR, 2),
            label=rf'Polya Efficiency ($1\leq\theta\leq2$), ENC = {noise}',
            alpha=1-noise/100
        )

    ax.grid()
    ax.legend()
    ax.set_xscale('log')
    ax.set_xlabel('Threshold / Noise: (TNR)')
    ax.set_ylabel('Occupancy / Efficiency')
    ax.set_title(rf'Gas Gain: $\bar n $ = {gain}')

plt.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

freq = 100e6

ENC = np.array([3, 60])

cf = 4.37e-15
echarge = 1.602e-19

vnoise = ENC*echarge/cf

threshold = np.logspace(-5, -1, 501)

for ax in axs:
    for noise in vnoise:
        TNR = threshold/noise
        riceRate = riceNoiseRate(TNR, freq)
    
        ax.plot(
            threshold*1e3, riceRate,
            label=f'V_noise = {noise*1e2:.3f} mV'
        )


    ax.set_xlabel('Threshold (mV)')
    ax.set_ylabel('Rice Noise Rate (Hz)')

    ax.set_yscale('log')
    ax.set_ylim([1, 1e9])

    ax.legend()
    ax.grid()

axs[0].set_xlim([0, 15])
axs[1].set_xscale('log')


plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 100e6
dt = 1266e-9

ENC = np.array([3, 60])
gains = [50, 1000]

threshold = np.logspace(-5, 0, 501)

cf = 4.37e-15
echarge = 1.602e-19

vnoise = ENC*echarge/cf

fig, axs = plt.subplots(1, 2, figsize=(12, 4))

for ax, gain in zip(axs, gains):

    for noise, enc in zip(vnoise, ENC):
        TNR = threshold/noise
        riceNoiseOccupancy = riceOccupancy(TNR, freq, dt)

        isPrimary = (gain, enc) in zip(gains, ENC)
        
        ax.plot(
            threshold*1e3, riceNoiseOccupancy,
            lw=2.5,
            ls='-' if isPrimary else '--', 
            alpha=1 if isPrimary else 0.25,
            label=f'ENC = {enc}'
        )
        
    GTR = gain*echarge/cf/threshold
    ax.fill_between(
        threshold*1e3, 
        myPolyaEfficiencyNormal(GTR, 1), 
        myPolyaEfficiencyNormal(GTR, 2),
        #label='Polya:' + '\n' + rf'  $\bar n $ = {gain:.0f}' + '\n' + r'  $1\leq\theta\leq2$',
        label=rf'Polya ($1\leq\theta\leq2$)',
        color='r', alpha=.75
    )

    ax.grid()
    ax.legend()
    ax.set_xscale('log')
    ax.set_xlabel('Threshold (mV)')
    ax.set_ylabel('Occupancy / Efficiency')
    ax.set_title(rf'Gas Gain: $\bar n = {gain}$')
    
plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 100e6
dt = 1266e-9

threshold = np.logspace(-5, 0, 501)

cf = 4.37e-15
echarge = 1.602e-19

configs = [
    {'name': 'FIMS', 'enc': 3,  'gain': 100, 'threshold': 10, 'ls': '-',  'c':'b', 'alpha': 1},
    {'name': 'GridPix', 'enc': 90, 'gain': 2500, 'threshold': 700, 'ls': '--', 'c':'r', 'alpha': 0.6},
    #{'name': 'GridPix', 'enc': 60, 'gain': 1000, 'threshold': 515, 'ls': '--', 'c':'r', 'alpha': 0.6},
]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

def mVtoElectron(x):
    return x*cf/echarge/1e3
def mVfromElectron(x):
    return x*echarge/cf*1e3
    
axElectron = ax.secondary_xaxis('top', functions=(mVtoElectron, mVfromElectron))
axElectron.set_xlabel('Threshold (# of Electrons)')

for cfg in configs:
    enc, gain = cfg['enc'], cfg['gain']

    noise = enc * echarge / cf
    TNR = threshold/noise
    riceNoiseOccupancy = riceOccupancy(TNR, freq, dt)
    
    ax.plot(
        threshold*1e3, riceNoiseOccupancy,
        lw=2.5, ls=cfg['ls'], alpha=cfg['alpha'], c=cfg['c'],
        label=f'{cfg['name']} Noise'
    )
    
    GTR = gain*echarge/cf/threshold
    ax.fill_between(
        threshold*1e3, 
        myPolyaEfficiencyNormal(GTR, 1), 
        myPolyaEfficiencyNormal(GTR, 2),
        #label=rf'Polya: $\overline{{n}} $ = {gain:.0f}, $1\leq\theta\leq2$',
        label=f'{cfg['name']} Signal',
        color=cfg['c'], alpha=0.5*cfg['alpha'], ls=cfg['ls']
    )

    ax.axvline(
        mVfromElectron(cfg['threshold']), 
        c=cfg['c'], alpha=.5, ls=':', lw=1,
        label=f'{cfg['name']} Threshold'
    )

#ax.grid()
ax.axhline(0, c='k', lw=1)
ax.axhline(1, c='k', lw=1)
ax.axhline(.95, c='k', ls='--')


ax.legend()
ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_xlim([.06, None])
#ax.set_ylim([None, 1.5])
ax.set_xlabel('Threshold (mV)')
ax.set_ylabel('Occupancy / Efficiency')
    
plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 100e6
dt = 1266e-9

threshold = np.logspace(0, 5, 501)

cf = 4.37e-15
echarge = 1.602e-19

configs = [
    {'name': 'FIMS', 'enc': 3,  'gain': 100, 'threshold': 10,  'c':'b', 'alpha': 1},
    #{'name': 'GridPix', 'enc': 90, 'gain': 2500, 'threshold': 700, 'c':'r', 'alpha': 0.6},
    #{'name': 'GridPix', 'enc': 60, 'gain': 1000, 'threshold': 515, 'ls': '--', 'c':'r', 'alpha': 0.6},
]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

for cfg in configs:
    enc, gain = cfg['enc'], cfg['gain']

    noise = enc
    TNR = threshold/noise
    riceNoiseOccupancy = riceOccupancy(TNR, freq, dt)
    
    ax.plot(
        TNR, riceNoiseOccupancy,
        lw=2.5, ls='-', alpha=cfg['alpha'], c=cfg['c'],
        label=f'{cfg['name']} Noise'
    )
    
    GTR = gain/threshold
    ax.plot(
        TNR, 
        myPolyaEfficiencyNormal(GTR, 1.5),
        #label=f'{cfg['name']} Signal',
        color=cfg['c'], alpha=0.75*cfg['alpha'], ls='--'
    )
    ax.fill_between(
        TNR, 
        myPolyaEfficiencyNormal(GTR, 1), 
        myPolyaEfficiencyNormal(GTR, 2),
        #label=rf'Polya: $\overline{{n}} $ = {gain:.0f}, $1\leq\theta\leq2$',
        label=f'{cfg['name']} Signal', ls='--',
        color=cfg['c'], alpha=0.5*cfg['alpha']
    )
    ax.axvline(
        cfg['threshold']/cfg['enc'], 
        c=cfg['c'], alpha=.5, ls=':', lw=1,
        label=f'{cfg['name']} Threshold'
    )

ax.axhline(0, c='k', lw=1)
ax.axhline(1, c='k', lw=1)
ax.axhline(.95, c='k', ls='--')

ax.legend()
ax.set_xscale('log')
#ax.set_yscale('log')
ax.set_xlim([.6, 1e3])
#ax.set_ylim([None, 1.5])
ax.set_xlabel('Threshold / Noise (TNR)')
ax.set_ylabel('Occupancy / Efficiency')
    
plt.tight_layout()
plt.show()

In [ ]:
# Setup
freq = 100e6
dt = 1266e-9

threshold = np.logspace(-5, 5, 501)

cf = 4.37e-15
echarge = 1.602e-19

configs = [
    {'name': 'FIMS', 'enc': 3,  'gain': 83, 'threshold': 23, 'ls': '-',  'c':'b', 'alpha': 1},
    {'name': 'GridPix', 'enc': 90, 'gain': 2500, 'threshold': 700, 'ls': '--', 'c':'r', 'alpha': 0.6},
    #{'name': 'GridPix', 'enc': 60, 'gain': 1000, 'threshold': 515, 'ls': '--', 'c':'r', 'alpha': 0.6},
]

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

for cfg in configs:
    enc, gain = cfg['enc'], cfg['gain']

    noise = enc
    TNR = threshold/noise
    riceNoiseOccupancy = riceOccupancy(TNR, freq, dt)
    
    GTR = gain/threshold
    ax.fill_between(
        riceNoiseOccupancy, 
        myPolyaEfficiencyNormal(GTR, 1), 
        myPolyaEfficiencyNormal(GTR, 2),
        #label=rf'Polya: $\overline{{n}} $ = {gain:.0f}, $1\leq\theta\leq2$',
        label=f'{cfg['name']} Signal',
        color=cfg['c'], alpha=0.5*cfg['alpha'], ls=cfg['ls']
    )



ax.legend()
ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_xlim([.06, None])
#ax.set_ylim([None, 1.5])
ax.set_xlabel('Noise Occupancy')
ax.set_ylabel('Detection Efficiency')
    
plt.tight_layout()
plt.show()

In [ ]:
prob = np.logspace(-6, 0, endpoint=False)
cf = 4.37e-15
echarge = 1.602e-19

fig, ax = plt.subplots()

ax.plot(getThreshold(prob)*1e3, prob)

def mVtoElectron(x):
    return x*cf/echarge/1e3
def mVfromElectron(x):
    return x*echarge/cf*1e3
    
axElectron = ax.secondary_xaxis('top', functions=(mVtoElectron, mVfromElectron))
axElectron.set_xlabel('Threshold (# Electrons)')

ax.set_xlabel('Threshold (mV)')
ax.set_ylabel('P(False) / Chip / Readout')

ax.set_yscale('log')

ax.grid()
plt.show()

In [ ]:
def addHex(ax, sideLength, centerX, centerY, color, style):
    hexX = np.array([1, 0.5, -0.5, -1, -0.5, 0.5, 1])
    hexY = np.array([0, 1, 1, 0, -1, -1, 0])
    xVals = sideLength*hexX + centerX
    yVals = sideLength*math.sqrt(3)/2*hexY + centerY
    ax.plot(xVals, yVals, c=color, ls=style)
    return

def addCenter(ax, pitch, pad):
    sideLength = pitch/math.sqrt(3)
    addHex(ax, sideLength, 0, 0, 'b', '-')
    addHex(ax, pad, 0, 0, 'm', '-')
    return

def addNeighbors(ax, pitch, pad):
    sideLength = pitch/math.sqrt(3)
    dx = pitch*math.sqrt(3)/2
    dy = pitch
    neighborX = dx*np.array([0, 1, 1, 0, -1, -1])
    neighborY = dy*np.array([1, 0.5, -0.5, -1, -0.5, 0.5])
    for inX, inY in zip(neighborX, neighborY):
        addHex(ax, sideLength, inX, inY, 'b', ':')
        addHex(ax, pad, inX, inY, 'm', ':')
    return

def addUnitCell(ax, pitch):
    cellX = pitch*math.sqrt(3)/2*np.array([0, 1, 1, 0, 0])
    cellY = pitch/2*np.array([0, 0, 1, 1, 0])
    ax.plot(cellX, cellY, c='g', ls='-', lw=2)
    return

def addHalfUnitCell(ax, pitch):
    cellX = pitch*math.sqrt(3)/2*np.array([0, 1, 1, 0, 0])/2
    cellY = pitch/2*np.array([0, 0, 1, 1, 0])
    ax.plot(cellX, cellY, c='c', ls='--', lw=2)
    return

def addTiling(ax, pitch):
    xLines = pitch*math.sqrt(3)/2*np.array([-1, 0, 1])
    yLines = pitch/2*np.array([-3, -2, -1, 0, 1, 2, 3])
    for inX in xLines:
        ax.axvline(inX, c='r', ls=':')
    for inY in yLines:
        ax.axhline(inY, c='r', ls=':')
    return


def addOptionalTiling(ax, pitch):
    '''
    vLines = pitch*math.sqrt(3)/2*np.array([-1, 1])
    hLines = pitch/2*np.array([-2, 2])
    for inX in vLines:
        ax.axvline(inX, c='c', ls='-', lw=2)
    for inY in hLines:
        ax.axhline(inY, c='c', ls='-', lw=2)
    '''
    xVals = pitch*math.sqrt(3)/2*np.array([-1, 1, 1, -1, -1])
    yVals = pitch*np.array([-1, -1, 1, 1, -1])
    ax.plot(xVals, yVals, ls='-', c='c', lw=2)
    return
    

In [ ]:
pitch = 55
pad = 15

fig = plt.figure()
ax = fig.add_subplot(111)

addCenter(ax, pitch, pad)
addNeighbors(ax, pitch, pad)

addUnitCell(ax, pitch)
#addHalfUnitCell(ax, pitch)
#addTiling(ax, pitch)
addOptionalTiling(ax, pitch)


ax.set_aspect('equal')
ax.grid()
plt.tight_layout()
plt.show()

In [ ]:
pitch = 55
pad = 15

fig = plt.figure()
ax = fig.add_subplot(111)

addCenter(ax, pitch, pad)
addNeighbors(ax, pitch, pad)

#addUnitCell(ax, pitch)
#addTiling(ax, pitch)
#addOptionalTiling(ax, pitch)

repeatX = np.array([0, 0])
repeatY = pitch*np.array([1, -1])
for inX, inY in zip(repeatX, repeatY):
    addHex(ax, sideLength, inX, inY, 'b', '-')
    addHex(ax, pad, inX, inY, 'm', '-')


ax.set_aspect('equal')
ax.grid()
plt.tight_layout()
plt.show()
    
    